

#### 1º - Instalação de Bibliotecas

 **boto3**
- É o SDK oficial da AWS para Python
- Serve para acessar serviços da Amazon:
  - S3
  - Lambda
  - DynamoDB
  - Entre outros

 **camelot-py**
- Biblioteca especializada em extração de tabelas de PDFs
- Principais funcionalidades:
  - Leitura de arquivos PDF
  - Detecção automática de tabelas
  - Conversão para DataFrame (formato semelhante a Excel/CSV)

 **pandas**
- Biblioteca padrão para análise de dados em Python
- Utilizada para:
  - Manipulação de tabelas
  - Tratamento de dados
  - Exportação para CSV, Excel e outros formatos


In [ ]:
!pip install boto3 camelot-py pandas
!pip install awscli


In [ ]:
!aws --version

In [ ]:
!aws configure

In [ ]:
!aws sts get-caller-identity


#### 2º - Importação das bibliotecas

    import boto3
        - baixar PDF de um bucket S3
        - listar arquivos na nuvem
        - enviar CSV processado de volta

    import camelot
        - ler PDFs com tabelas
        - extrair dados estruturados (tipo planilha)
        - converter tabelas do PDF em DataFrame
    
    import pandas as pd
        - manipular tabelas (DataFrame)
        - limpar dados
        - salvar CSV

    import re
        - biblioteca de expressões regulares
        - limpar textos
        - remover caracteres estranhos do PDF
        - extrair padrões (ex: números, datas, nomes)

    import os
        - trabalha em pastas e arquivos
        - criar diretórios
        - verificar se arquivos existem
        - trabalhar com caminhos

    

In [ ]:
import boto3
import camelot
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', None)

import re
import os
import io

#### 3º - Conexão com AWS S3

    - Define as credenciais de autenticação
    - S3 → objeto que fala com a AWS
    - bucket → pasta onde estão os PDFs
    Colocando as credencias:

        Será solicitado:
            - access_key
            - secret_key
            - region
            - output format

In [ ]:
!aws configure list

In [ ]:
s3 = boto3.client("s3")

In [ ]:
s3 = boto3.client("s3")

response = s3.list_buckets()
print(response["Buckets"])

In [ ]:
s3.list_objects_v2(Bucket="tbh-resultados-raw")

#### 4º - Transformar o nome do arquivo em dados estruturados

        - Obtem os metadados da corrida e que serão adicionados na estrutura da tabela posteriormente.
        - Dados obtidos:
             - nome da corrida,
             - ano
             - genero
             - distancia da prova


In [ ]:
def extrair_metadados(key):
    ano = re.search(r'(\d{4})', key)
    ano = int(ano.group(1)) if ano else None

    filename = os.path.basename(key).replace('.pdf', '')
    partes = filename.split('_')

    corrida = []
    genero = None

    for p in partes:
        if p.upper() in ['FEMININO', 'MASCULINO']:
            genero = p.upper()
            break
        corrida.append(p)

    corrida = '_'.join(corrida)
    corrida = re.sub(r'_\d+.*$', '', corrida)

    distancia = None

    km_match = re.search(r'(\d+_\d+|\d+)\s*KM', filename.upper())

    if km_match:
        km_str = km_match.group(1)
        km_str = km_str.replace('_', '.')
        distancia = float(km_str)

    return {
        'ano': ano,
        'corrida': corrida,
        'genero': genero,
        'distancia_km': distancia
    }

##### GUIA : Identificar o nome nome dos arquivos no S3


In [ ]:
bucket = "tbh-resultados-raw"
prefix = "TBH Esportes/resultados_tbh_2026_completo/"

In [ ]:
def listar_arquivos_s3(s3, bucket, prefixo=None):
    arquivos = []

    kwargs = {'Bucket': bucket}
    if prefixo:
        kwargs['Prefix'] = prefixo

    while True:
        response = s3.list_objects_v2(**kwargs)

        for obj in response.get('Contents', []):
            arquivos.append(obj['Key'])

        # verifica se tem mais páginas
        if response.get('IsTruncated'):
            kwargs['ContinuationToken'] = response['NextContinuationToken']
        else:
            break

    return arquivos

In [ ]:
keys = listar_arquivos_s3(s3, bucket)

print("Total de arquivos:", len(keys))

print("\nPrimeiros 5 arquivos:\n")
for i, k in enumerate(keys[:5], 1):
    print(f"{i} - {k}")

##### EXEMPLO : Obter o nome de 1 arquivo para validação

In [ ]:
key = "TBH Esportes/resultados_tbh_2026_completo/corridadoger_10KM_FEMININO_PUBLICO_GERAL"

###### VALIDAÇÃO : Verificar como os dados referente a corrida estão retornando

In [ ]:
extrair_metadados(key)

#### 5º - Coleta os arquivos do S3 e retorna apenas o que é PDF

  
    - Percorre todos os arquivos do bucket/pasta no S3
    - Lida com paginação (caso tenha muitos arquivos)
    - Filtra apenas PDFs
    - Guarda tudo numa lista - pdf_keys



In [ ]:
paginator = s3.get_paginator('list_objects_v2')

pdf_keys = []

for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    contents = page.get('Contents', [])

    for obj in contents:
        key = obj['Key']

        # garante que só pega PDFs reais (evita zip etc.)
        if key.lower().endswith('.pdf'):
            pdf_keys.append(key)

print(f"Total de PDFs encontrados: {len(pdf_keys)}")

#### 6º - Normalização dos cabeçalhos

In [ ]:
arquivos_pdf = []

for key in pdf_keys:
    obj = s3.get_object(Bucket=bucket, Key=key)
    pdf_bytes = obj['Body'].read()

    arquivos_pdf.append({
        "key": key,
        "pdf_bytes": pdf_bytes
    })

print("Arquivos carregados:", len(arquivos_pdf))

In [ ]:
from tqdm import tqdm
cabecalhos = []

for arq in tqdm(arquivos_pdf, desc="Processando PDFs"):
    try:
        obj = s3.get_object(Bucket=bucket, Key=arq["key"])
        pdf_bytes = obj["Body"].read()

        tables = camelot.read_pdf(io.BytesIO(pdf_bytes))

        if len(tables) == 0:
            continue

        df = tables[0].df
        header = tuple(df.iloc[0].astype(str).str.strip().tolist())

        cabecalhos.append({
            "key": arq["key"],
            "cabecalho": header
        })

    except Exception as e:
        print("Erro no arquivo:", arq["key"], e)


In [ ]:
df.columns = df.iloc[0]
df = df[1:].reset_index(drop=True)

In [ ]:
from collections import Counter

contagem = Counter([item["cabecalho"] for item in cabecalhos])

print("Tipos de cabeçalho:", len(contagem))

In [ ]:
for i, (cab, qtd) in enumerate(contagem.items()):
    print(f"\n=== Tipo {i+1} ({qtd} arquivos) ===")
    print(cab)

In [ ]:
# confirmar a variacao de 6 arquivos
len(set(cabecalhos[i]["cabecalho"] for i in range(len(cabecalhos))))

In [ ]:
#08/05
standard_header = (
    'Coloc.', 'Num.', 'Nome', 'Sx.', 'Idd.', 'Faixa', 'Cl.Fx.', 'Equipe', 'Tempo', 'Liquido'
)

In [ ]:
#08/05
current_header_tipo6 = (
    'Coloc.', 'Numero', 'Nome', 'Id.', 'Fx.Et.', 'Cl.Fx.', 'Sx.', 'C.', 'Equipe', 'Tempo'
)

In [ ]:
#08/05

def normalize_tipo6(df):
    # 1. Definir o cabeçalho padrão (alvo)
    standard_header = [
        'Coloc.', 'Num.', 'Nome', 'Sx.', 'Idd.', 'Faixa', 'Cl.Fx.', 'Equipe', 'Tempo', 'Liquido'
    ]

    # 2. Dicionário de renomeação específica para o Tipo 6
    rename_rules = {
        'Numero': 'Num.',
        'Id.': 'Idd.',
        'Fx.Et.': 'Faixa'
    }

    # 3. Aplicar a renomeação
    df = df.rename(columns=rename_rules)

    # 4. Reconstruir o DataFrame na ordem correta
    # Isso garante que:
    # - Colunas extras (como 'C.') sejam removidas
    # - Colunas faltantes (como 'Liquido') sejam criadas com valores vazios
    # - A ordem das colunas seja exatamente a do standard_header
    normalized_df = pd.DataFrame(columns=standard_header)

    for col in standard_header:
        if col in df.columns:
            normalized_df[col] = df[col]
        else:
            normalized_df[col] = None

    return normalized_df

# --- Exemplo de Execução ---

# Simulando dados do Tipo 6
dados = [
    ['1', '500', 'JOÃO SILVA', '25', 'M2029', '1', 'M', 'X', 'EQUIPE A', '00:20:00'],
    ['2', '501', 'MARIA SOUZA', '30', 'F3039', '1', 'F', 'Y', 'EQUIPE B', '00:22:15']
]
colunas_originais = ['Coloc.', 'Numero', 'Nome', 'Id.', 'Fx.Et.', 'Cl.Fx.', 'Sx.', 'C.', 'Equipe', 'Tempo']

df_original = pd.DataFrame(dados, columns=colunas_originais)

# Normalizando
df_final = normalize_tipo6(df_original)

print("Cabeçalho após normalização:")
print(df_final.columns.tolist())
print("\nConteúdo normalizado:")
print(df_final)

In [ ]:
key_real = "TBH Esportes/resultados_tbh_2026_completo/alllimits_CORRIDA_HIGH_LIMITS_FEMININO.pdf"

def normalize_tipo1_corrigido(df_bruto_camelot):
    """Normaliza um DataFrame com cabeçalho Tipo 1 para o cabeçalho padrão, com correção de extração."""
    standard_header = [
        'Coloc.', 'Num.', 'Nome', 'Sx.', 'Idd.', 'Faixa', 'Cl.Fx.', 'Equipe', 'Tempo', 'Liquido'
    ]

    # A primeira linha do df_bruto_camelot contém os cabeçalhos originais
    # e as demais linhas contêm os dados.
    # Vamos primeiro definir os cabeçalhos com base na primeira linha do DataFrame bruto
    # e depois processar os dados.

    # Cria uma cópia para não modificar o DataFrame original passado como argumento
    df = df_bruto_camelot.copy()

    # Define a primeira linha como cabeçalho e remove-a dos dados
    df.columns = df.iloc[0].astype(str).str.strip()
    df = df[1:].reset_index(drop=True)

    # Renomear a coluna vazia (índice 2) para 'Nome' e a coluna 'Num Atleta' para 'Num.'
    # e 'F.E' para 'Cl.Fx.'
    # A coluna 'C.' será descartada na reordenação
    rename_rules = {
        'Pos': 'Coloc.',
        'Num Atleta': 'Num.',
        '': 'Nome', # Mapeia a coluna vazia para 'Nome'
        'F.E': 'Cl.Fx.'
    }
    df = df.rename(columns=rename_rules)

    # Limpar quebras de linha na coluna 'Nome'
    if 'Nome' in df.columns:
        df['Nome'] = df['Nome'].astype(str).str.replace('\n', ' ', regex=False).str.strip()

    # Reconstruir o DataFrame na ordem padrão
    normalized_df = pd.DataFrame(columns=standard_header)
    for col in standard_header:
        if col in df.columns:
            normalized_df[col] = df[col]
        else:
            normalized_df[col] = None # Adiciona colunas faltantes com None

    return normalized_df

def testar_normalizacao_tipo1_real():
    print(f"--- Iniciando extração e normalização corrigida do arquivo real: {key_real} ---\n")

    try:
        # Extração do PDF via S3
        obj = s3.get_object(Bucket=bucket, Key=key_real)
        pdf_bytes = obj["Body"].read()

        # Leitura das tabelas com Camelot
        # Usamos flavor='lattice' que é bom para tabelas com linhas visíveis
        # Se não funcionar, pode tentar flavor='stream'
        tables = camelot.read_pdf(io.BytesIO(pdf_bytes), flavor='lattice')

        if len(tables) == 0:
            print("⚠️ Tentando com flavor='stream'...")
            tables = camelot.read_pdf(io.BytesIO(pdf_bytes), flavor='stream')

        if len(tables) == 0:
            print("❌ Nenhuma tabela encontrada no PDF.")
            return

        df_bruto_camelot = tables[0].df

        print("✅ Cabeçalho bruto extraído pelo Camelot (primeira linha do DF):")
        print(list(df_bruto_camelot.iloc[0].astype(str).str.strip()))

        # APLICAÇÃO DA NORMALIZAÇÃO CORRIGIDA
        df_normalizado = normalize_tipo1_corrigido(df_bruto_camelot)

        print("\n✅ Cabeçalho após normalização corrigida:")
        print(list(df_normalizado.columns))

        print("\n--- Primeiras 5 linhas do dado normalizado ---")
        display(df_normalizado.head()) # No Colab, use display() para ver a tabela formatada

    except Exception as e:
        print(f"❌ Erro durante o processamento: {e}")

# Executar o teste
testar_normalizacao_tipo1_real()